<style>

.footnote-list {
    display: none;
}
</style>

# Basic BPE Tokenizer
A trained BPE takes an input text, splits it up into tokens, and assigns a token ID (not to be confused with the numerical vector embeddings of said token):

```{figure} ../../figures/class1/004_BPE.png
---
name: BPE-high-level
---
Modified from [Sebastian Rasckha](https://github.com/rasbt/LLMs-from-scratch/blob/main) under the [APACHE License](https://github.com/rasbt/LLMs-from-scratch/blob/main/LICENSE.txt).

This is what we'll aim to do!

## 4.1 Training
We begin the process with an initial vocabulary that is a set of all individual characters[^special_ex]

Then we do these these steps:
1. **Count** the most frequent pairs of existing tokens (**characters** *or* **bytes**[^bytes_ex]) in our corpus 
2. **Merge** that pair (to become a new token)
3. **Replace** that pair with a token ID
3. **Repeat** 1 & 2 until there are no gains 
    - Or until we've hit "max" vocabulary length, set by a parameter *vocab_size* or *k* in [Jurafsky & Martin](https://web.stanford.edu/~jurafsky/slp3/2.pdf).


:::{admonition} "Pairs" Clarification
:class: important
I use "pair" to mean **two adjacent tokens**. At the start, this is may be individual characters or bytes[^pairs_ex] (e.g., the pair "t" + "h"), but they quickly become combinations beyond single characters or bytes (e.g., "th" + "e" which can be merged to form"the").
:::

[^special_ex]: Some implementations add "special character tokens". We won't do this here. See [Wiki/BPE](https://en.wikipedia.org/wiki/Byte-pair_encoding#Modified_algorithm)
[^bytes_ex]: LLMs use Byte-level BPE, not character-level. We'll start with words for intution, then transition to bytes.
[^pairs_ex]: At byte-level, it is even more complicated, since some Unicode characters, such as "Ø", are represented by more than 1 byte. See [Encode Text section](#encode-text).

### Step 1: Finding Frequent Combinations
Let's start with an example of a simple sequence:


In [157]:
input_text = "the cat sat on the mat"

In Python, we can easily iterate over characters in a string. Let's do that with `enumerate` where we can also get the position of the character: 

In [158]:
for i, char in enumerate(input_text):
    print(i, char)

0 t
1 h
2 e
3  
4 c
5 a
6 t
7  
8 s
9 a
10 t
11  
12 o
13 n
14  
15 t
16 h
17 e
18  
19 m
20 a
21 t


#### Construct Pairs
We need to construct pairs. That is, go through each `char` and combine with `next_char`. By hand, this for loop looks like this:
```
iteration 1: 
    char = t 
    next_char = h
    pair = (char, next_char)
iteration 2: 
    char = h 
    next_char = e
    pair = (char, next_char)
```

:::{admonition} HANDS-ON: Identify the `next_char` & create pairs!
:class: red
Loop over characters with `enumerate`: 
1. Identify `char` and `next_char`
2. Save these as a pair 
3. Print the pair
4. Remember to end the for loop *before* you check the last character, since that one won't have a consecutive pair to check with!
:::: 

##### Solution

:::{admonition} How to identify `next_char`
:class: tip, dropdown
Each character in a string has a position. We can select each character with this position e.g., text[2] would be "e"[^zero_index]. With `enumerate`, we've made that position explicit using `i, char`. `i` is the current char, how would we get the next one?

<details>
<summary>Click to see answer</summary>
We identify the <code>next_char</code> by adding +1 to <code>i</code>, i.e., <code>text[i + 1]</code>
</details>
::::

[^zero_index]: Remember that Python is zero-indexed!

:::{admonition} End the loop before the last character.
:class: tip, dropdown
The problem is that we are comparing consecutive characters, and that 
::::

In [159]:
for i, char in enumerate(input_text[:-1]):
    next_char = input_text[i + 1]
    pair = (char, next_char)
    print(pair)

('t', 'h')
('h', 'e')
('e', ' ')
(' ', 'c')
('c', 'a')
('a', 't')
('t', ' ')
(' ', 's')
('s', 'a')
('a', 't')
('t', ' ')
(' ', 'o')
('o', 'n')
('n', ' ')
(' ', 't')
('t', 'h')
('h', 'e')
('e', ' ')
(' ', 'm')
('m', 'a')
('a', 't')


#### Count Pairs
We shouldn't just print the pairs, but record each unique pair and the frequency with which they occur to a `dictionary`. Below is a starting point:

In [160]:
def count_pairs(input_data):
    """"
    A function which takes input_data (string or byte-encoded), constructs a dictionary of unique consecutive pairs 
    & counts their occurence in input data.
    """
    # initialize empty counts dictoinary
    counts = {} 

    # your code

    return counts

:::{admonition} HANDS-ON
:class: red
Finish the function above. 

Your function should loop over `input_text`, constructing pairs and counting them in a `counts` dictionary. 

This dictionary has each **unique pair** as a `key` and their frequency as a `value`. In other words, this is the end goal:
```
counts = {('t', 'h'): 2, ('h', 'e'): 2, ('e', ' '): 2, (' ', 'c'): 1, ('c', 'a'): 1, ('a', 't'): 3, ...}
```
:::

##### Solution

:::{admonition} HINT. Incrementing a value in a dictionary
:class: tip, dropdown
See this [geeksforgeeks](https://www.geeksforgeeks.org/python/python-increment-value-in-dictionary/) guide on ways to increment values in a dictionary!
::::

In [168]:
def count_pairs(input_data):
    """"
    A function which takes input_data (string or byte-encoded), constructs a dictionary of unique consecutive pairs 
    & counts their occurence in input data.
    """
    # initialize empty counts dictoinary
    counts = {} 

    # loop over each char in input text except the last one!
    for i, char in enumerate(input_data[:-1]):
        next_char = input_data[i + 1]
        pair = (char, next_char)

        try: # try to increment a value to a key that already exists
            counts[pair] += 1 
        except KeyError: # if key does not exist, it'll throw a key error (you won't be able to increment it). But you can create it!
            counts[pair] = 1

    return counts

#### Encode Text!

The example above used *characters*. Computers represent these with the **Unicode** standard, which assigns a unique number (**a code point**) to each character, no matter language or symbol. More than 150,000 codepoints[^uni_explorer] exist. For example, the Danish Å, Ø, Æ are represented as:
```
Å = U+00C5
Ø = U+00D8
Æ = U+00C6
```

Rather than initializing BPE with a vocabulary of +150000 code points, we encode our text into **UTF-8**, an encoding that stores these characters as **single bytes** or **sequences of bytes**:
```python
Å = [195, 133] # 2 bytes!
Ø = [195, 152]
Æ = [195, 134]

A = [65] # 1 byte!
```

Each byte has up to 256 possible values [^optional_ex], so a natural starting point for (Byte-level) BPE is these values as its initial tokens. Above token number 255[^zero_index_byte], we begin creating tokens by merging byte-sequences!

:::{admonition} OPTIONAL: Why 256 values? What is a byte actually?
:class: tip, dropdown
A *byte* is a series of 8 **bits**, where each bit can hold a binary number 0 or 1 (e.g., 0110 0110). This means a single byte can represent $2^8 = 256$ possible *decimal* values, ranging from 0 to 255. 

Let's break this down. Think of each bit as a light switch holding 2 possible values:
```
bit 1:  0 or 1
bit 2:  0 or 1
bit 3:  0 or 1
bit 4:  0 or 1
bit 5:  0 or 1
bit 6:  0 or 1
bit 7:  0 or 1
bit 8:  0 or 1
```

With 2 possible values for each bit, we get 256 possible combinations for 8 bits:
```
2 × 2 × 2 × 2 × 2 × 2 × 2 × 2 = 2^8 = 256
```

We can also represent these positiotns as powers of 2, starting at 2^0 on the right: 
| Position | 7 | 6 | 5 | 4 | 3 | 2 | 1 | 0 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Power of 2 | \(2^7\) | \(2^6\) | \(2^5\) | \(2^4\) | \(2^3\) | \(2^2\) | \(2^1\) | \(2^0\) |
| Value | 128 | 64 | 32 | 16 | 8 | 4 | 2 | 1 |

For example, the decimal value "1" is represented in binary as:
```
0 0 0 0 0 0 0 1
```
The 1 is in the 2^0 position, and 2^0 is equal to 1.

If you want to geek about about binary systems and how to convert from decimal numbers (e.g,. 120) (eg., 0110 0110). I highly recommend these videos from Khan Academy:
1. [The binary system](https://www.khanacademy.org/computing/computers-and-internet/xcae6f4a7ff015e7d:digital-information/xcae6f4a7ff015e7d:binary-numbers/v/the-binary-number-system?referrer=share_link)
2. [Convert decimial numbers to binary](https://www.khanacademy.org/computing/computers-and-internet/xcae6f4a7ff015e7d:digital-information/xcae6f4a7ff015e7d:binary-numbers/v/converting-decimal-numbers-to-binary?referrer=share_link).
::::

[^uni_explorer]: You can explore these codepoints on [unicode-explorer.com](https://unicode-explorer.com/).
[^optional_ex]: I initally went on a *long* tangent about bits & bytes here, but decided against it. Read more about it in the optional tip box.
[^zero_index_byte]: Again, we're in the zero index world, so values from 0 to 255 = 256!

**In code**, we simple convert text by using the **encode("utf.8)** method on our string. We wrap it in a list to get the bytes out:

In [169]:
input_bytes = list(input_text.encode("utf-8"))
print(input_bytes)

[116, 104, 101, 32, 99, 97, 116, 32, 115, 97, 116, 32, 111, 110, 32, 116, 104, 101, 32, 109, 97, 116]


Our count function will work just fine, even with bits!

In [170]:
count_pairs(input_bytes)

{(116, 104): 2,
 (104, 101): 2,
 (101, 32): 2,
 (32, 99): 1,
 (99, 97): 1,
 (97, 116): 3,
 (116, 32): 2,
 (32, 115): 1,
 (115, 97): 1,
 (32, 111): 1,
 (111, 110): 1,
 (110, 32): 1,
 (32, 116): 1,
 (32, 109): 1,
 (109, 97): 1}

##### Decode (revert back)
If we want to check this, we can use the `chr()` to **decode** the bytes. Here for the first byte-pair:

In [175]:
print(chr(116))
print(chr(104))

t
h


## References
This NB is partially inspired by Sebastian Rasckha's "Build a Large Language Model from Scratch" and HF's [BPE tutorial](https://huggingface.co/learn/llm-course/en/chapter6/5), both licensed under APACHE.

``